# Model Comparison: RGB to NIR/RedEdge Reconstruction

Compare 4 models side-by-side:
1. **WaveletMST** - DWT + Spectral Attention
2. **CNN_MSTPP_Decoder** - CNN Stem + MST++ Blocks
3. **MSTPP** - Multi-Stage Spectral-wise Transformer (Cai et al. CVPRW 2022)
4. **MicroNIR** - Ultra-lightweight Inverted Residual Network

## Metrics (aligned with Remote Sensing 2025 paper):
- **MRAE** (Mean Relative Absolute Error) - Primary metric
- **RMSE** (Root Mean Square Error)
- **MAE** (Mean Absolute Error)
- **SAM** (Spectral Angle Mapper) - Spectral fidelity
- **SID** (Spectral Information Divergence)
- **NDVI MAE** / **NDRE MAE** - Vegetation index accuracy

## 1. Setup and Imports

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# Mount Google Drive (Colab)
try:
    from google.colab import drive, auth
    drive.mount('/content/drive')
    auth.authenticate_user()
    print("Google Drive mounted")
except Exception as e:
    print(f"Not running on Colab or Drive already mounted: {e}")

In [ ]:
class CompareConfig:
    """Common configuration for comparison."""

    # GCS Data
    BUCKET = 'YOUR_BUCKET'
    FOLDER = 'dataset_training_mst'
    PATCH_SIZE = 128
    INPUT_CHANNELS = 3
    OUTPUT_CHANNELS = 4

    # Output constraints
    OUTPUT_MIN = 0.0
    OUTPUT_MAX = 1.0

    # Batch size for evaluation
    BATCH_SIZE = 16

    # Model weights paths (adjust to your Drive paths)
    WAVELET_WEIGHTS = '/content/drive/MyDrive/WaveletMST_Checkpoints/wavelet_mst_best.weights.h5'
    CNN_MSGPP_WEIGHTS = '/content/drive/MyDrive/CNN_MSGPP_Checkpoints/cnn_msgpp_decoder_best.weights.h5'
    MSTPP_WEIGHTS = '/content/drive/MyDrive/MSTPP_Checkpoints/mstpp_best.weights.h5'
    MICRONIR_WEIGHTS = '/content/drive/MyDrive/MicroNIR_Checkpoints/micronir_best.weights.h5'

config = CompareConfig()
print("Configuration loaded")

## 2. Common Layers and Utilities

In [ ]:
from tensorflow.keras import layers
import math

keras = tf.keras

# ============================================================================
# COMMON UTILITIES
# ============================================================================

def get_activation(name):
    activations = {
        'relu': 'relu', 'gelu': 'gelu', 'swish': 'swish',
        'silu': 'swish', 'mish': 'mish'
    }
    if name.lower() not in activations:
        raise ValueError(f"Unknown activation: {name}. Choices: {list(activations.keys())}")
    return activations[name.lower()]

def get_activation_layer(name):
    activation_name = get_activation(name)
    return layers.Activation(activation_name)

def get_initializer(activation='relu'):
    if activation in ['relu', 'leaky_relu', 'elu']:
        return tf.keras.initializers.HeNormal()
    return tf.keras.initializers.GlorotUniform()


class RMSNorm(layers.Layer):
    def __init__(self, eps=1e-6, **kwargs):
        super().__init__(**kwargs)
        self.eps = eps

    def build(self, input_shape):
        self.gamma = self.add_weight(
            name='gamma', shape=(input_shape[-1],),
            initializer='ones', trainable=True
        )

    def call(self, x):
        x_float = tf.cast(x, tf.float32)
        rms = tf.sqrt(tf.reduce_mean(tf.square(x_float), axis=-1, keepdims=True) + self.eps)
        x_normed = x_float / rms
        gamma = tf.cast(self.gamma, x_float.dtype)
        result = x_normed * gamma
        return tf.cast(result, x.dtype)

    def get_config(self):
        config = super().get_config()
        config.update({'eps': self.eps})
        return config


class DropPath(layers.Layer):
    def __init__(self, drop_prob=0.1, **kwargs):
        super().__init__(**kwargs)
        self.drop_prob = drop_prob

    def call(self, x, training=None):
        if not training or self.drop_prob == 0:
            return x
        keep_prob = 1 - self.drop_prob
        shape = (tf.shape(x)[0],) + (1,) * (len(x.shape) - 1)
        random_tensor = keep_prob + tf.random.uniform(shape, dtype=x.dtype)
        binary_mask = tf.floor(random_tensor)
        return x / keep_prob * binary_mask

    def get_config(self):
        config = super().get_config()
        config.update({'drop_prob': self.drop_prob})
        return config

print("Common utilities defined")

## 3. Model Architectures

In [ ]:
# ============================================================================
# WAVELET MST ARCHITECTURE (exact copy from Train_WaveletMST.ipynb)
# ============================================================================

class HaarWavelet2D(layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):
        sqrt2 = np.sqrt(2)
        L = np.array([1, 1]) / sqrt2
        H = np.array([1, -1]) / sqrt2

        self.filter_LL = self.add_weight(
            name='filter_LL', shape=(2, 2, 1, 1),
            initializer=tf.constant_initializer(np.outer(L, L).reshape(2, 2, 1, 1)),
            trainable=False, dtype='float32')
        self.filter_LH = self.add_weight(
            name='filter_LH', shape=(2, 2, 1, 1),
            initializer=tf.constant_initializer(np.outer(L, H).reshape(2, 2, 1, 1)),
            trainable=False, dtype='float32')
        self.filter_HL = self.add_weight(
            name='filter_HL', shape=(2, 2, 1, 1),
            initializer=tf.constant_initializer(np.outer(H, L).reshape(2, 2, 1, 1)),
            trainable=False, dtype='float32')
        self.filter_HH = self.add_weight(
            name='filter_HH', shape=(2, 2, 1, 1),
            initializer=tf.constant_initializer(np.outer(H, H).reshape(2, 2, 1, 1)),
            trainable=False, dtype='float32')
        super().build(input_shape)

    def call(self, x):
        x_f32 = tf.cast(x, tf.float32)
        C = tf.shape(x_f32)[-1]
        results = {}
        filters = {
            'LL': self.filter_LL, 'LH': self.filter_LH,
            'HL': self.filter_HL, 'HH': self.filter_HH
        }
        for name, filt in filters.items():
            filt_expanded = tf.tile(filt, [1, 1, C, 1])
            out = tf.nn.depthwise_conv2d(x_f32, filt_expanded, strides=[1, 2, 2, 1], padding='SAME')
            results[name] = tf.cast(out, x.dtype)
        return results


class MultiLevelDWT(layers.Layer):
    def __init__(self, num_levels=2, **kwargs):
        super().__init__(**kwargs)
        self.num_levels = num_levels

    def build(self, input_shape):
        self.dwt_layers = [HaarWavelet2D(name=f'dwt_level_{i}') for i in range(self.num_levels)]
        super().build(input_shape)

    def call(self, x):
        all_details = []
        current = x
        for level, dwt in enumerate(self.dwt_layers):
            coeffs = dwt(current)
            all_details.append({
                'level': level, 'LH': coeffs['LH'],
                'HL': coeffs['HL'], 'HH': coeffs['HH'], 'LL': coeffs['LL']
            })
            current = coeffs['LL']
        return all_details, current


class DWTFeatureExtractor(layers.Layer):
    def __init__(self, num_levels=2, output_dim=64, activation='swish', **kwargs):
        super().__init__(**kwargs)
        self.num_levels = num_levels
        self.output_dim = output_dim
        self.activation_name = get_activation(activation)
        self.initializer = get_initializer(activation)

    def build(self, input_shape):
        self.dwt = MultiLevelDWT(self.num_levels)

        detail_dim = max(self.output_dim // (self.num_levels + 1), 16)
        approx_dim = self.output_dim - (detail_dim * self.num_levels)

        self.detail_projs = []
        self.detail_activations = []
        self.detail_upsamplers = []

        for level in range(self.num_levels):
            self.detail_projs.append(
                layers.Conv2D(detail_dim, 1, kernel_initializer=self.initializer,
                              name=f'detail_proj_level_{level}'))
            self.detail_activations.append(
                layers.Activation(self.activation_name, name=f'detail_act_level_{level}'))
            scale_factor = 2 ** (level + 1)
            self.detail_upsamplers.append(
                layers.UpSampling2D(size=scale_factor, interpolation='nearest'))

        self.approx_proj = layers.Conv2D(
            max(approx_dim, 16), 1,
            kernel_initializer=self.initializer, name='approx_proj')
        self.approx_activation = layers.Activation(self.activation_name, name='approx_act')
        self.approx_upsampler = layers.UpSampling2D(
            size=2**self.num_levels, interpolation='bilinear')

        self.fusion = layers.Conv2D(
            self.output_dim, 1,
            kernel_initializer=self.initializer, name='dwt_fusion')
        super().build(input_shape)

    def call(self, x):
        input_shape = tf.shape(x)
        H, W = input_shape[1], input_shape[2]

        details_by_level, final_approx = self.dwt(x)
        features = []

        for level, (detail_proj, detail_act, upsampler) in enumerate(
            zip(self.detail_projs, self.detail_activations, self.detail_upsamplers)
        ):
            level_details = details_by_level[level]
            detail_concat = tf.concat([
                level_details['LH'], level_details['HL'], level_details['HH']
            ], axis=-1)
            projected = detail_proj(detail_concat)
            projected = detail_act(projected)
            upsampled = upsampler(projected)
            features.append(upsampled)

        approx_proj = self.approx_proj(final_approx)
        approx_proj = self.approx_activation(approx_proj)
        approx_upsampled = self.approx_upsampler(approx_proj)
        features.append(approx_upsampled)

        features_aligned = [tf.image.resize(f, [H, W], method='nearest') for f in features]
        all_features = tf.concat(features_aligned, axis=-1)
        output = self.fusion(all_features)
        return output


# ============================================================================
# TRUE SPECTRAL ATTENTION (correct version from training notebooks)
# ============================================================================

class TrueSpectralAttention(layers.Layer):
    """True Spectral Attention optimized for memory.
    STABLE VERSION: Critical computations in float32."""
    def __init__(self, dim, num_heads, qk_norm=True, **kwargs):
        super().__init__(**kwargs)
        if dim % num_heads != 0:
            raise ValueError(f"dim ({dim}) must be divisible by num_heads ({num_heads})")
        self.num_heads = num_heads
        self.dim = dim
        self.head_dim = dim // num_heads
        self.qk_norm = qk_norm

    def build(self, input_shape):
        init = tf.keras.initializers.TruncatedNormal(stddev=0.02)
        self.qkv = layers.Dense(self.dim * 3, use_bias=False,
                                kernel_initializer=init, name='qkv_proj')
        self.proj = layers.Dense(self.dim, kernel_initializer=init, name='out_proj')
        if self.qk_norm:
            self.q_norm = RMSNorm(name='q_norm')
            self.k_norm = RMSNorm(name='k_norm')
        super().build(input_shape)

    def call(self, x, training=False):
        input_dtype = x.dtype
        B = tf.shape(x)[0]
        H = tf.shape(x)[1]
        W = tf.shape(x)[2]
        C = self.dim
        N = tf.cast(H, tf.int64) * tf.cast(W, tf.int64)
        N_i32 = tf.cast(N, tf.int32)

        # Reshape: [B, H, W, C] -> [B, N, C]
        x_flat = tf.reshape(x, (B, N_i32, C))

        # QKV projection on C dimension
        qkv = self.qkv(x_flat)  # [B, N, 3*C]
        qkv = tf.reshape(qkv, (B, N_i32, 3, C))
        q, k, v = qkv[:, :, 0], qkv[:, :, 1], qkv[:, :, 2]

        # QK Normalization
        if self.qk_norm:
            q = self.q_norm(q)
            k = self.k_norm(k)

        # Split heads: [B, N, C] -> [B, heads, N, head_dim]
        q = tf.reshape(q, (B, N_i32, self.num_heads, self.head_dim))
        k = tf.reshape(k, (B, N_i32, self.num_heads, self.head_dim))
        v = tf.reshape(v, (B, N_i32, self.num_heads, self.head_dim))
        q = tf.transpose(q, (0, 2, 1, 3))
        k = tf.transpose(k, (0, 2, 1, 3))
        v = tf.transpose(v, (0, 2, 1, 3))

        # SPECTRAL ATTENTION in float32 for stability
        q_f32 = tf.cast(q, tf.float32)
        k_f32 = tf.cast(k, tf.float32)
        v_f32 = tf.cast(v, tf.float32)

        # Q: [B, heads, N, head_dim] -> [B, heads, head_dim, N]
        q_t = tf.transpose(q_f32, (0, 1, 3, 2))

        # Spectral attention: [B, heads, head_dim, N] @ [B, heads, N, head_dim]
        # -> [B, heads, head_dim, head_dim]   (C×C attention, NOT N×N!)
        scale_n = tf.math.rsqrt(tf.cast(N, tf.float32))
        attn = tf.matmul(q_t, k_f32) * scale_n

        # Stable softmax
        attn = attn - tf.reduce_max(attn, axis=-1, keepdims=True)
        attn = tf.nn.softmax(attn, axis=-1)

        # V: [B, heads, N, head_dim] -> [B, heads, head_dim, N]
        v_t = tf.transpose(v_f32, (0, 1, 3, 2))

        # Output: [B, heads, head_dim, head_dim] @ [B, heads, head_dim, N]
        # -> [B, heads, head_dim, N]
        out = tf.matmul(attn, v_t)

        # Back: [B, heads, head_dim, N] -> [B, heads, N, head_dim]
        out = tf.transpose(out, (0, 1, 3, 2))
        out = tf.reshape(out, (B, N_i32, C))

        out = tf.cast(out, input_dtype)
        out = self.proj(out)
        out = tf.reshape(out, (B, H, W, C))
        return out


class ChannelAttention(layers.Layer):
    def __init__(self, channels, reduction=8, activation='swish', **kwargs):
        super().__init__(**kwargs)
        self.channels = channels
        self.reduction = reduction
        self.activation_name = get_activation(activation)

    def build(self, input_shape):
        init = get_initializer('swish')
        reduced_channels = max(self.channels // self.reduction, 4)
        self.fc1 = layers.Dense(reduced_channels, kernel_initializer=init)
        self.activation = layers.Activation(self.activation_name)
        self.fc2 = layers.Dense(self.channels,
                                kernel_initializer=tf.keras.initializers.GlorotNormal())
        super().build(input_shape)

    def call(self, x):
        gap = tf.reduce_mean(x, axis=[1, 2], keepdims=True)
        squeeze = tf.squeeze(gap, axis=[1, 2])
        excite = self.fc1(squeeze)
        excite = self.activation(excite)
        excite = self.fc2(excite)
        excite = tf.nn.sigmoid(excite)
        excite = tf.reshape(excite, (-1, 1, 1, self.channels))
        return x * excite


class SpatialAttention(layers.Layer):
    def __init__(self, kernel_size=7, **kwargs):
        super().__init__(**kwargs)
        self.kernel_size = kernel_size

    def build(self, input_shape):
        self.conv = layers.Conv2D(
            1, self.kernel_size, padding='same',
            kernel_initializer='glorot_uniform', use_bias=False)
        super().build(input_shape)

    def call(self, x):
        avg_pool = tf.reduce_mean(x, axis=-1, keepdims=True)
        max_pool = tf.reduce_max(x, axis=-1, keepdims=True)
        concat = tf.concat([avg_pool, max_pool], axis=-1)
        attention = tf.nn.sigmoid(self.conv(concat))
        return x * attention


class SpatialMixerDWConv(layers.Layer):
    def __init__(self, channels, kernel_size=7, expansion=2, drop_rate=0.0, **kwargs):
        super().__init__(**kwargs)
        self.channels = channels
        self.kernel_size = kernel_size
        self.expansion = expansion
        self.drop_rate = drop_rate

    def build(self, input_shape):
        self.norm = RMSNorm(name='norm')
        self.dwconv = layers.DepthwiseConv2D(
            self.kernel_size, padding='same',
            depthwise_initializer='he_normal', name='dwconv')
        self.pwconv_expand = layers.Conv2D(
            self.channels * self.expansion, 1, padding='same',
            kernel_initializer='he_normal', name='pwconv_expand')
        self.act = layers.Activation('gelu', name='gelu')
        self.pwconv_project = layers.Conv2D(
            self.channels, 1, padding='same',
            kernel_initializer='he_normal', name='pwconv_project')
        self.drop = layers.Dropout(self.drop_rate) if self.drop_rate > 0 else None
        super().build(input_shape)

    def call(self, x, training=False):
        y = self.norm(x)
        y = self.dwconv(y)
        y = self.pwconv_expand(y)
        y = self.act(y)
        y = self.pwconv_project(y)
        if self.drop is not None:
            y = self.drop(y, training=training)
        return y


class CBAM(layers.Layer):
    def __init__(self, channels, reduction=8, **kwargs):
        super().__init__(**kwargs)
        self.channels = channels
        self.reduction = reduction

    def build(self, input_shape):
        self.channel_att = ChannelAttention(self.channels, self.reduction)
        self.spatial_att = SpatialAttention()
        super().build(input_shape)

    def call(self, x):
        x = self.channel_att(x)
        x = self.spatial_att(x)
        return x


class SpectralTransformerBlock(layers.Layer):
    """Memory-optimized Transformer block with Pre-Norm, RMSNorm, and DropPath."""
    def __init__(self, embed_dim, num_heads, mlp_ratio=4, dropout=0.1,
                 drop_path=0.0, activation='gelu',
                 use_spatial_mixing=False, spatial_kernel=7, spatial_expansion=2,
                 use_channel_attention=False, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.mlp_ratio = mlp_ratio
        self.dropout_rate = dropout
        self.drop_path_rate = drop_path
        self.activation_name = activation
        self.use_spatial_mixing = use_spatial_mixing
        self.spatial_kernel = spatial_kernel
        self.spatial_expansion = spatial_expansion
        self.use_channel_attention = use_channel_attention

    def build(self, input_shape):
        self.norm1 = RMSNorm(name='norm1')
        self.norm2 = RMSNorm(name='norm2')

        if self.use_spatial_mixing:
            self.spatial_mix = SpatialMixerDWConv(
                self.embed_dim,
                kernel_size=self.spatial_kernel,
                expansion=self.spatial_expansion,
                drop_rate=self.dropout_rate,
                name='spatial_mix')

        self.spectral_attn = TrueSpectralAttention(
            self.embed_dim, self.num_heads, qk_norm=True)

        if self.use_channel_attention:
            self.channel_attn = ChannelAttention(self.embed_dim, activation='swish')

        self.drop_path = DropPath(self.drop_path_rate) if self.drop_path_rate > 0 else lambda x, training: x

        init = get_initializer(self.activation_name)
        mlp_hidden = self.embed_dim * self.mlp_ratio

        self.mlp = keras.Sequential([
            layers.Dense(mlp_hidden, kernel_initializer=init),
            layers.Activation(get_activation(self.activation_name)),
            layers.Dropout(self.dropout_rate),
            layers.Dense(self.embed_dim, kernel_initializer=init),
            layers.Dropout(self.dropout_rate)
        ], name='mlp')

        super().build(input_shape)

    def call(self, x, training=False):
        if self.use_spatial_mixing:
            x = x + self.drop_path(self.spatial_mix(x, training=training), training=training)

        normed = self.norm1(x)
        attn_out = self.spectral_attn(normed, training=training)
        if self.use_channel_attention:
            attn_out = self.channel_attn(attn_out)
        x = x + self.drop_path(attn_out, training=training)

        x = x + self.drop_path(self.mlp(self.norm2(x), training=training), training=training)
        return x


def build_wavelet_mst(config):
    """Build WaveletMST model (exact match to training notebook)."""
    inputs = keras.Input(shape=(config.PATCH_SIZE, config.PATCH_SIZE, config.INPUT_CHANNELS))

    init = get_initializer(config.ACTIVATION)
    activation_name = get_activation(config.ACTIVATION)

    # Direct RGB -> NIR projection (global skip connection)
    rgb_to_nir = layers.Conv2D(
        config.OUTPUT_CHANNELS, 1, padding='same', use_bias=True,
        kernel_initializer='glorot_uniform',
        bias_initializer=tf.keras.initializers.Constant(0.1),
        dtype='float32', name='rgb_to_nir_proj'
    )(inputs)

    # Branch 1: Multi-level DWT
    dwt_features = DWTFeatureExtractor(
        num_levels=config.DWT_LEVELS,
        output_dim=config.EMBED_DIM // 2,
        activation=config.ACTIVATION,
        name='dwt_encoder'
    )(inputs)

    # Branch 2: Shallow CNN
    shallow = layers.Conv2D(64, 3, padding='same', kernel_initializer=init, name='shallow_1')(inputs)
    shallow = layers.Activation(activation_name, name='shallow_act_1')(shallow)
    shallow = layers.Conv2D(config.EMBED_DIM // 2, 3, padding='same', kernel_initializer=init, name='shallow_2')(shallow)
    shallow = layers.Activation(activation_name, name='shallow_act_2')(shallow)

    # Fusion DWT + CNN
    fused = layers.Concatenate(name='dwt_cnn_fusion')([dwt_features, shallow])
    fused = layers.Conv2D(config.EMBED_DIM, 1, kernel_initializer=init, name='fusion_proj')(fused)
    fused = layers.Activation(activation_name, name='fusion_act')(fused)

    # Spectral Transformer blocks
    x = fused
    num_blocks = config.NUM_TRANSFORMER_BLOCKS
    dpr = [i * config.DROP_PATH_RATE / max(num_blocks - 1, 1) for i in range(num_blocks)]

    for i in range(num_blocks):
        x = SpectralTransformerBlock(
            embed_dim=config.EMBED_DIM,
            num_heads=config.NUM_HEADS,
            mlp_ratio=config.MLP_RATIO,
            dropout=config.DROPOUT,
            drop_path=dpr[i],
            activation='gelu',
            use_spatial_mixing=getattr(config, 'USE_SPATIAL_MIXING', False),
            spatial_kernel=getattr(config, 'SPATIAL_KERNEL', 7),
            spatial_expansion=getattr(config, 'SPATIAL_EXPANSION', 2),
            use_channel_attention=getattr(config, 'USE_CHANNEL_ATTENTION', False),
            name=f'spectral_block_{i}'
        )(x)

    # Final norm + local skip
    x = RMSNorm(name='final_norm')(x)
    x = layers.Add(name='local_skip')([x, fused])

    # Decoder
    x = layers.Conv2D(64, 3, padding='same', kernel_initializer=init, name='refine_1')(x)
    x = RMSNorm(name='refine_norm_1')(x)
    x = layers.Activation(activation_name, name='refine_act_1')(x)

    if getattr(config, 'USE_DECODER_ATTENTION', False):
        x = CBAM(64, reduction=8, name='decoder_cbam')(x)

    x = layers.Conv2D(32, 3, padding='same', kernel_initializer=init, name='refine_2')(x)
    x = RMSNorm(name='refine_norm_2')(x)
    x = layers.Activation(activation_name, name='refine_act_2')(x)

    residual = layers.Conv2D(
        config.OUTPUT_CHANNELS, 3, padding='same',
        kernel_initializer=tf.keras.initializers.TruncatedNormal(stddev=0.01),
        bias_initializer='zeros', dtype='float32', name='residual_output'
    )(x)

    outputs = layers.Add(name='final_output')([rgb_to_nir, residual])

    output_constraint = getattr(config, 'OUTPUT_CONSTRAINT', None)
    if output_constraint == 'clip':
        outputs = layers.Lambda(
            lambda t: tf.clip_by_value(t, config.OUTPUT_MIN, config.OUTPUT_MAX),
            name='output_clip')(outputs)
    elif output_constraint == 'tanh':
        outputs = layers.Lambda(
            lambda t: (0.5 * (tf.tanh(t) + 1.0)) * (config.OUTPUT_MAX - config.OUTPUT_MIN) + config.OUTPUT_MIN,
            name='output_tanh')(outputs)

    model = keras.Model(inputs=inputs, outputs=outputs, name='WaveletMST_v1.5')
    return model

print("WaveletMST architecture defined")

In [ ]:
# ============================================================================
# CNN_MSGPP_DECODER ARCHITECTURE (exact copy from Train_CNN_MSGPP_Decoder_L1_SAM.ipynb)
# ============================================================================

class MSGPPMixBlock(layers.Layer):
    """MST++/ViT-style block with spatial mixing, spectral attention, and MLP."""
    def __init__(self, channels, num_heads, mlp_ratio=4, drop_rate=0.0, drop_path=0.0,
                 spatial_kernel=7, spatial_expansion=2, **kwargs):
        super().__init__(**kwargs)
        self.channels = channels
        self.num_heads = num_heads
        self.mlp_ratio = mlp_ratio
        self.drop_rate = drop_rate
        self.drop_path_rate = drop_path
        self.spatial_kernel = spatial_kernel
        self.spatial_expansion = spatial_expansion

    def build(self, input_shape):
        init = get_initializer('gelu')

        self.norm_spatial = RMSNorm(name='norm_spatial')
        self.norm_attn = RMSNorm(name='norm_attn')
        self.norm_mlp = RMSNorm(name='norm_mlp')

        self.dwconv = layers.DepthwiseConv2D(
            self.spatial_kernel, padding='same',
            depthwise_initializer='he_normal', name='dwconv')
        self.pw_expand = layers.Conv2D(
            self.channels * self.spatial_expansion, 1, padding='same',
            kernel_initializer='he_normal', name='pw_expand')
        self.spatial_act = layers.Activation('gelu', name='spatial_gelu')
        self.pw_project = layers.Conv2D(
            self.channels, 1, padding='same',
            kernel_initializer='he_normal', name='pw_project')

        self.spectral_attn = TrueSpectralAttention(
            self.channels, self.num_heads, qk_norm=True, name='spectral_attn')

        mlp_hidden = self.channels * self.mlp_ratio
        self.mlp_expand = layers.Conv2D(mlp_hidden, 1, padding='same',
            kernel_initializer=init, name='mlp_expand')
        self.mlp_act = layers.Activation('gelu', name='mlp_gelu')
        self.mlp_project = layers.Conv2D(self.channels, 1, padding='same',
            kernel_initializer=init, name='mlp_project')
        self.mlp_drop = layers.Dropout(self.drop_rate) if self.drop_rate > 0 else None

        self.drop_path = DropPath(self.drop_path_rate) if self.drop_path_rate > 0 else None
        super().build(input_shape)

    def _apply_drop_path(self, x, training=False):
        if self.drop_path is None:
            return x
        return self.drop_path(x, training=training)

    def call(self, x, training=False):
        # Spatial mixing
        y = self.norm_spatial(x)
        y = self.dwconv(y)
        y = self.pw_expand(y)
        y = self.spatial_act(y)
        y = self.pw_project(y)
        x = x + self._apply_drop_path(y, training=training)

        # Spectral attention
        y = self.norm_attn(x)
        y = self.spectral_attn(y, training=training)
        x = x + self._apply_drop_path(y, training=training)

        # MLP/FFN
        y = self.norm_mlp(x)
        y = self.mlp_expand(y)
        y = self.mlp_act(y)
        if self.mlp_drop is not None:
            y = self.mlp_drop(y, training=training)
        y = self.mlp_project(y)
        if self.mlp_drop is not None:
            y = self.mlp_drop(y, training=training)
        x = x + self._apply_drop_path(y, training=training)

        return x


def build_cnn_msgpp_decoder(config):
    """Build CNN_MSGPP_Decoder model (exact match to training notebook)."""
    inputs = keras.Input(shape=(config.PATCH_SIZE, config.PATCH_SIZE, config.INPUT_CHANNELS))

    init = get_initializer(config.ACTIVATION)
    activation_name = get_activation(config.ACTIVATION)

    # Direct RGB -> NIR projection (global skip connection)
    rgb_to_nir = layers.Conv2D(
        config.OUTPUT_CHANNELS, 1, padding='same', use_bias=True,
        kernel_initializer='glorot_uniform',
        bias_initializer=tf.keras.initializers.Constant(0.1),
        dtype='float32', name='rgb_to_nir_proj'
    )(inputs)

    # CNN Stem
    x = layers.Conv2D(64, 3, padding='same', kernel_initializer=init, name='stem_conv_1')(inputs)
    if getattr(config, 'STEM_RMSNORM', True):
        x = RMSNorm(name='stem_norm_1')(x)
    x = layers.Activation(activation_name, name='stem_act_1')(x)

    x = layers.Conv2D(config.BASE_DIM, 3, padding='same', kernel_initializer=init, name='stem_conv_2')(x)
    if getattr(config, 'STEM_RMSNORM', True):
        x = RMSNorm(name='stem_norm_2')(x)
    x = layers.Activation(activation_name, name='stem_act_2')(x)

    # MSGPP blocks
    num_blocks = config.NUM_MSGPP_BLOCKS
    dpr = [i * config.DROP_PATH_RATE / max(num_blocks - 1, 1) for i in range(num_blocks)]

    for i in range(num_blocks):
        x = MSGPPMixBlock(
            channels=config.BASE_DIM,
            num_heads=config.NUM_HEADS,
            mlp_ratio=config.MLP_RATIO,
            drop_rate=config.DROPOUT,
            drop_path=dpr[i],
            spatial_kernel=getattr(config, 'SPATIAL_KERNEL', 7),
            spatial_expansion=getattr(config, 'SPATIAL_EXPANSION', 2),
            name=f'msgpp_block_{i}'
        )(x)

    x = RMSNorm(name='final_norm')(x)

    # Convolutional decoder
    x = layers.Conv2D(64, 3, padding='same', kernel_initializer=init, name='refine_1')(x)
    x = RMSNorm(name='refine_norm_1')(x)
    x = layers.Activation(activation_name, name='refine_act_1')(x)

    x = layers.Conv2D(32, 3, padding='same', kernel_initializer=init, name='refine_2')(x)
    x = RMSNorm(name='refine_norm_2')(x)
    x = layers.Activation(activation_name, name='refine_act_2')(x)

    residual = layers.Conv2D(
        config.OUTPUT_CHANNELS, 3, padding='same',
        kernel_initializer=tf.keras.initializers.TruncatedNormal(stddev=0.01),
        bias_initializer=tf.keras.initializers.Constant(-0.6689549),
        dtype='float32', name='residual_output'
    )(x)

    outputs = layers.Add(name='final_output')([rgb_to_nir, residual])

    output_constraint = getattr(config, 'OUTPUT_CONSTRAINT', None)
    if output_constraint == 'clip':
        outputs = layers.Lambda(
            lambda t: tf.clip_by_value(t, config.OUTPUT_MIN, config.OUTPUT_MAX),
            name='output_clip')(outputs)
    elif output_constraint == 'tanh':
        outputs = layers.Lambda(
            lambda t: (0.5 * (tf.tanh(t) + 1.0)) * (config.OUTPUT_MAX - config.OUTPUT_MIN) + config.OUTPUT_MIN,
            name='output_tanh')(outputs)

    model = keras.Model(inputs=inputs, outputs=outputs, name='CNN_MSGPP_Decoder_v1')
    return model

print("CNN_MSGPP_Decoder architecture defined")

In [ ]:
# ============================================================================
# MSTPP ARCHITECTURE (exact copy from Models_ipynb/Train_MSTPP_New.ipynb)
# ============================================================================

class SMSA(layers.Layer):
    """Spectral-wise Multi-head Self-Attention (S-MSA).
    Computes channel attention (C×C) by treating each channel as a token."""
    def __init__(self, dim, num_heads=1, bias=False, **kwargs):
        super().__init__(**kwargs)
        self.dim = dim
        self.num_heads = num_heads
        self.bias = bias
        self.head_dim = dim // num_heads

    def build(self, input_shape):
        self.qkv = layers.Conv2D(self.dim * 3, 1, use_bias=self.bias,
                                  kernel_initializer='glorot_uniform', name='qkv')
        self.proj = layers.Conv2D(self.dim, 1, use_bias=self.bias,
                                   kernel_initializer='glorot_uniform', name='proj')
        super().build(input_shape)

    def call(self, x, training=None):
        shape = tf.shape(x)
        B, H, W, C = shape[0], shape[1], shape[2], shape[3]
        qkv = self.qkv(x)
        qkv = tf.reshape(qkv, [B, H * W, 3, self.num_heads, self.head_dim])
        qkv = tf.transpose(qkv, [2, 0, 3, 4, 1])
        q, k, v = qkv[0], qkv[1], qkv[2]
        seq_len = tf.cast(H * W, tf.float32)
        scale = 1.0 / seq_len
        attn = tf.matmul(q, k, transpose_b=True) * scale
        attn = tf.clip_by_value(attn, -30.0, 30.0)
        attn = tf.nn.softmax(attn, axis=-1)
        out = tf.matmul(attn, v)
        out = tf.transpose(out, [0, 3, 1, 2])
        out = tf.reshape(out, [B, H, W, C])
        out = self.proj(out)
        return out


class GDFN(layers.Layer):
    """Gated-Dconv Feed-Forward Network."""
    def __init__(self, dim, ffn_expansion=2.66, bias=False, **kwargs):
        super().__init__(**kwargs)
        self.dim = dim
        self.hidden_dim = int(dim * ffn_expansion)
        self.bias = bias

    def build(self, input_shape):
        self.project_in = layers.Conv2D(self.hidden_dim * 2, 1, use_bias=self.bias,
                                         kernel_initializer='glorot_uniform', name='project_in')
        self.dwconv = layers.DepthwiseConv2D(3, padding='same', use_bias=self.bias,
                                              depthwise_initializer='glorot_uniform', name='dwconv')
        self.project_out = layers.Conv2D(self.dim, 1, use_bias=self.bias,
                                          kernel_initializer='glorot_uniform', name='project_out')
        super().build(input_shape)

    def call(self, x, training=None):
        x = self.project_in(x)
        x1, x2 = tf.split(x, 2, axis=-1)
        x1 = tf.clip_by_value(x1, -10.0, 10.0)
        x2 = tf.clip_by_value(x2, -10.0, 10.0)
        x = self.dwconv(x1) * tf.nn.gelu(x2)
        x = tf.clip_by_value(x, -10.0, 10.0)
        x = self.project_out(x)
        return x


class SAB(layers.Layer):
    """Spectral-wise Attention Block: LayerNorm → S-MSA → + → LayerNorm → GDFN → +"""
    def __init__(self, dim, num_heads=1, ffn_expansion=2.66, bias=False, **kwargs):
        super().__init__(**kwargs)
        self.dim = dim
        self.num_heads = num_heads
        self.ffn_expansion = ffn_expansion
        self.bias = bias

    def build(self, input_shape):
        self.norm1 = layers.LayerNormalization(epsilon=1e-5, name='norm1')
        self.attn = SMSA(self.dim, self.num_heads, self.bias, name='attn')
        self.norm2 = layers.LayerNormalization(epsilon=1e-5, name='norm2')
        self.ffn = GDFN(self.dim, self.ffn_expansion, self.bias, name='ffn')
        super().build(input_shape)

    def call(self, x, training=None):
        x = x + 0.1 * self.attn(self.norm1(x), training=training)
        x = x + 0.1 * self.ffn(self.norm2(x), training=training)
        return x


class MSTDownsample(layers.Layer):
    """Pixel unshuffle for downsampling."""
    def __init__(self, dim, **kwargs):
        super().__init__(**kwargs)
        self.dim = dim

    def build(self, input_shape):
        self.conv = layers.Conv2D(self.dim * 2, 1, use_bias=False,
                                   kernel_initializer='glorot_uniform', name='conv')
        super().build(input_shape)

    def call(self, x, training=None):
        shape = tf.shape(x)
        B, H, W, C = shape[0], shape[1], shape[2], shape[3]
        x = tf.reshape(x, [B, H // 2, 2, W // 2, 2, C])
        x = tf.transpose(x, [0, 1, 3, 2, 4, 5])
        x = tf.reshape(x, [B, H // 2, W // 2, C * 4])
        x = self.conv(x)
        return x


class MSTUpsample(layers.Layer):
    """Pixel shuffle for upsampling."""
    def __init__(self, dim, **kwargs):
        super().__init__(**kwargs)
        self.dim = dim

    def build(self, input_shape):
        self.conv = layers.Conv2D(self.dim * 2, 1, use_bias=False,
                                   kernel_initializer='glorot_uniform', name='conv')
        super().build(input_shape)

    def call(self, x, training=None):
        x = self.conv(x)
        x = tf.nn.depth_to_space(x, 2)
        return x


class SST(layers.Layer):
    """Single-stage Spectral-wise Transformer (U-shaped encoder-decoder)."""
    def __init__(self, dim, num_blocks, num_heads, ffn_expansion=2.66, **kwargs):
        super().__init__(**kwargs)
        self.dim = dim
        self.num_blocks = num_blocks
        self.num_heads = num_heads
        self.ffn_expansion = ffn_expansion

    def build(self, input_shape):
        self.enc0 = [SAB(self.dim, self.num_heads[0], self.ffn_expansion, name=f'enc0_sab{i}')
                     for i in range(self.num_blocks[0])]
        self.down0 = MSTDownsample(self.dim, name='down0')

        self.enc1 = [SAB(self.dim * 2, self.num_heads[1], self.ffn_expansion, name=f'enc1_sab{i}')
                     for i in range(self.num_blocks[1])]
        self.down1 = MSTDownsample(self.dim * 2, name='down1')

        self.bottleneck = [SAB(self.dim * 4, self.num_heads[2], self.ffn_expansion, name=f'bottleneck_sab{i}')
                           for i in range(self.num_blocks[2])]

        self.up1 = MSTUpsample(self.dim * 4, name='up1')
        self.reduce1 = layers.Conv2D(self.dim * 2, 1, use_bias=False, name='reduce1')
        self.dec1 = [SAB(self.dim * 2, self.num_heads[1], self.ffn_expansion, name=f'dec1_sab{i}')
                     for i in range(self.num_blocks[1])]

        self.up0 = MSTUpsample(self.dim * 2, name='up0')
        self.reduce0 = layers.Conv2D(self.dim, 1, use_bias=False, name='reduce0')
        self.dec0 = [SAB(self.dim, self.num_heads[0], self.ffn_expansion, name=f'dec0_sab{i}')
                     for i in range(self.num_blocks[0])]
        super().build(input_shape)

    def call(self, x, training=None):
        enc0_out = x
        for blk in self.enc0:
            enc0_out = blk(enc0_out, training=training)

        enc1_in = self.down0(enc0_out, training=training)
        enc1_out = enc1_in
        for blk in self.enc1:
            enc1_out = blk(enc1_out, training=training)

        bottleneck_in = self.down1(enc1_out, training=training)
        bottleneck_out = bottleneck_in
        for blk in self.bottleneck:
            bottleneck_out = blk(bottleneck_out, training=training)

        dec1_in = self.up1(bottleneck_out, training=training)
        dec1_in = self.reduce1(tf.concat([dec1_in, enc1_out], axis=-1))
        dec1_out = dec1_in
        for blk in self.dec1:
            dec1_out = blk(dec1_out, training=training)

        dec0_in = self.up0(dec1_out, training=training)
        dec0_in = self.reduce0(tf.concat([dec0_in, enc0_out], axis=-1))
        dec0_out = dec0_in
        for blk in self.dec0:
            dec0_out = blk(dec0_out, training=training)
        return dec0_out


class MSTPP(layers.Layer):
    """MST++ (Multi-Stage Spectral-wise Transformer). Cascades multiple SST stages."""
    def __init__(self, config, **kwargs):
        super().__init__(**kwargs)
        self.config = config
        self.dim = config.DIM
        self.num_stages = config.NUM_STAGES
        self.num_blocks = config.NUM_BLOCKS
        self.num_heads = config.NUM_HEADS
        self.ffn_expansion = config.FFN_EXPANSION

    def build(self, input_shape):
        self.input_proj = layers.Conv2D(self.dim, 3, padding='same', use_bias=False,
                                         kernel_initializer='glorot_uniform', name='input_proj')
        self.stages = [SST(self.dim, self.num_blocks, self.num_heads,
                           self.ffn_expansion, name=f'sst_stage{i}')
                       for i in range(self.num_stages)]
        self.output_proj = layers.Conv2D(self.config.OUTPUT_CHANNELS, 3, padding='same', use_bias=True,
                                          kernel_initializer=tf.initializers.TruncatedNormal(stddev=0.02),
                                          bias_initializer='zeros', name='output_proj')
        super().build(input_shape)

    def call(self, x, training=None):
        x = self.input_proj(x)
        for stage in self.stages:
            x = stage(x, training=training) + x
        x = self.output_proj(x)
        x = tf.nn.sigmoid(x)
        return x


def build_mstpp(config):
    """Build MST++ model (exact match to Train_MSTPP_New.ipynb)."""
    inputs = keras.Input(shape=(config.PATCH_SIZE, config.PATCH_SIZE, config.INPUT_CHANNELS))
    outputs = MSTPP(config, name='mstpp')(inputs)
    model = keras.Model(inputs, outputs, name='MSTPP')
    return model

print("MSTPP architecture defined")

In [ ]:
# ============================================================================
# MICRONIR ARCHITECTURE (exact copy from Models_ipynb/Train_MicroNIR.ipynb)
# ============================================================================

class SqueezeExcitation(layers.Layer):
    """Squeeze-and-Excitation block (Hu et al., 2018).

    Global channel attention at negligible parameter cost.
    For 64 channels with ratio=4: only 2×(64×16 + 16) = 2,080 params.
    """
    def __init__(self, channels, ratio=4, **kwargs):
        super().__init__(**kwargs)
        self.channels = channels
        self.ratio = ratio

    def build(self, input_shape):
        reduced = max(self.channels // self.ratio, 4)
        self.fc1 = layers.Dense(reduced, kernel_initializer='he_normal')
        self.fc2 = layers.Dense(self.channels, kernel_initializer='glorot_normal')
        super().build(input_shape)

    def call(self, x):
        # Global Average Pooling
        gap = tf.reduce_mean(x, axis=[1, 2], keepdims=False)  # (B, C)
        # Squeeze
        s = self.fc1(gap)
        s = tf.nn.swish(s)
        # Excitation
        s = self.fc2(s)
        s = tf.nn.sigmoid(s)
        # Scale
        s = tf.reshape(s, (-1, 1, 1, self.channels))
        return x * s

    def get_config(self):
        config = super().get_config()
        config.update({'channels': self.channels, 'ratio': self.ratio})
        return config


class InvertedResidualBlock(layers.Layer):
    """Inverted Residual Block (MobileNetV2 + BSConv style).

    Pattern: Expand (1×1) → Depthwise (3×3) → Project (1×1) + Skip

    For channels=64, expansion=2:
      - Expand: 64→128 (1×1) = 8,320 params
      - DWConv: 128ch (3×3) = 1,280 params
      - Project: 128→64 (1×1) = 8,256 params
      - Total: ~18K params (vs ~74K for standard 2× Conv3×3)
    """
    def __init__(self, channels, expansion=2, drop_path_rate=0.0, **kwargs):
        super().__init__(**kwargs)
        self.channels = channels
        self.expansion = expansion
        self.drop_path_rate = drop_path_rate

    def build(self, input_shape):
        expanded = self.channels * self.expansion

        # Pre-norm
        self.norm = RMSNorm(name='pre_norm')

        # Expand (pointwise)
        self.expand_conv = layers.Conv2D(
            expanded, 1, padding='same',
            kernel_initializer='he_normal',
            use_bias=False, name='expand'
        )
        self.expand_norm = RMSNorm(name='expand_norm')
        self.expand_act = layers.Activation('gelu', name='expand_act')

        # Depthwise spatial mixing
        self.dwconv = layers.DepthwiseConv2D(
            3, padding='same',
            depthwise_initializer='he_normal',
            use_bias=False, name='dwconv'
        )
        self.dw_norm = RMSNorm(name='dw_norm')
        self.dw_act = layers.Activation('gelu', name='dw_act')

        # Project (pointwise) - no activation after projection
        self.project_conv = layers.Conv2D(
            self.channels, 1, padding='same',
            kernel_initializer='glorot_uniform',
            use_bias=False, name='project'
        )

        # DropPath for regularization
        self.drop_path = DropPath(self.drop_path_rate) if self.drop_path_rate > 0 else None

        super().build(input_shape)

    def call(self, x, training=False):
        residual = x

        # Pre-norm → Expand → DWConv → Project
        y = self.norm(x)
        y = self.expand_conv(y)
        y = self.expand_norm(y)
        y = self.expand_act(y)

        y = self.dwconv(y)
        y = self.dw_norm(y)
        y = self.dw_act(y)

        y = self.project_conv(y)

        # DropPath + Skip connection
        if self.drop_path is not None:
            y = self.drop_path(y, training=training)

        return residual + y

    def get_config(self):
        config = super().get_config()
        config.update({
            'channels': self.channels,
            'expansion': self.expansion,
            'drop_path_rate': self.drop_path_rate,
        })
        return config


def build_micronir(config):
    """Build MicroNIR: ultra-lightweight RGB → NIR model.

    Architecture:
      RGB → Stem(Conv3×3) → N× InvRes → SE → Refine → Output
       │                                                  ↑
       └──────── Conv1×1 (global residual) ──────────────┘
    """
    inputs = keras.Input(shape=(config.PATCH_SIZE, config.PATCH_SIZE, config.INPUT_CHANNELS))

    init = get_initializer(config.ACTIVATION)
    activation_name = get_activation(config.ACTIVATION)

    # ======== Global Skip: Direct RGB → NIR projection ========
    rgb_to_nir = layers.Conv2D(
        config.OUTPUT_CHANNELS, 1,
        padding='same',
        use_bias=True,
        kernel_initializer='glorot_uniform',
        bias_initializer=tf.keras.initializers.Constant(0.1),
        dtype='float32',
        name='rgb_to_nir_proj'
    )(inputs)

    # ======== Shallow Feature Extraction (Stem) ========
    x = layers.Conv2D(
        config.BASE_DIM, 3, padding='same',
        kernel_initializer=init, name='stem_conv'
    )(inputs)
    x = RMSNorm(name='stem_norm')(x)
    x = layers.Activation(activation_name, name='stem_act')(x)

    # ======== Inverted Residual Blocks ========
    num_blocks = config.NUM_BLOCKS
    dpr = [i * config.DROP_PATH_RATE / max(num_blocks - 1, 1) for i in range(num_blocks)]

    for i in range(num_blocks):
        x = InvertedResidualBlock(
            channels=config.BASE_DIM,
            expansion=config.EXPANSION_RATIO,
            drop_path_rate=dpr[i],
            name=f'invres_block_{i}'
        )(x)

    # ======== Post-blocks normalization ========
    x = RMSNorm(name='final_norm')(x)

    # ======== Squeeze-and-Excitation (global channel recalibration) ========
    x = SqueezeExcitation(
        config.BASE_DIM,
        ratio=config.SE_RATIO,
        name='se_block'
    )(x)

    # ======== Lightweight Decoder ========
    x = layers.Conv2D(
        32, 3, padding='same',
        kernel_initializer=init, name='refine_conv'
    )(x)
    x = RMSNorm(name='refine_norm')(x)
    x = layers.Activation(activation_name, name='refine_act')(x)

    # ======== Residual Output ========
    residual = layers.Conv2D(
        config.OUTPUT_CHANNELS, 3,
        padding='same',
        kernel_initializer=tf.keras.initializers.TruncatedNormal(stddev=0.01),
        bias_initializer=tf.keras.initializers.Constant(-0.6689549),
        dtype='float32',
        name='residual_output'
    )(x)

    # Output = RGB projection + learned residual
    outputs = layers.Add(name='final_output')([rgb_to_nir, residual])

    # Output constraint
    output_constraint = getattr(config, 'OUTPUT_CONSTRAINT', None)
    if output_constraint == 'clip':
        outputs = layers.Lambda(
            lambda t: tf.clip_by_value(t, config.OUTPUT_MIN, config.OUTPUT_MAX),
            name='output_clip'
        )(outputs)
    elif output_constraint == 'tanh':
        outputs = layers.Lambda(
            lambda t: (0.5 * (tf.tanh(t) + 1.0)) * (config.OUTPUT_MAX - config.OUTPUT_MIN) + config.OUTPUT_MIN,
            name='output_tanh'
        )(outputs)

    model = keras.Model(inputs=inputs, outputs=outputs, name='MicroNIR_v1')
    return model

print("MicroNIR architecture defined")

## 4. Model Configs

In [ ]:
# Model-specific configs (matching training notebooks exactly)
class WaveletConfig(CompareConfig):
    EMBED_DIM = 112
    NUM_HEADS = 8
    NUM_TRANSFORMER_BLOCKS = 5
    MLP_RATIO = 3
    DROPOUT = 0.1
    DROP_PATH_RATE = 0.15
    DWT_LEVELS = 2
    ACTIVATION = 'swish'
    USE_SPATIAL_MIXING = True
    USE_CHANNEL_ATTENTION = False
    SPATIAL_KERNEL = 7
    SPATIAL_EXPANSION = 2
    USE_DECODER_ATTENTION = True
    OUTPUT_CONSTRAINT = 'tanh'

class CNNMSGPPConfig(CompareConfig):
    BASE_DIM = 112
    NUM_HEADS = 8
    NUM_MSGPP_BLOCKS = 4
    MLP_RATIO = 4
    DROPOUT = 0.1
    DROP_PATH_RATE = 0.15
    ACTIVATION = 'gelu'
    SPATIAL_KERNEL = 7
    SPATIAL_EXPANSION = 2
    STEM_RMSNORM = True
    OUTPUT_CONSTRAINT = 'tanh'

class MSTPPConfig(CompareConfig):
    DIM = 28
    NUM_STAGES = 3
    NUM_BLOCKS = [2, 2, 2]
    NUM_HEADS = [1, 2, 4]
    FFN_EXPANSION = 2.66
    # MSTPP uses sigmoid output, no separate constraint needed

class MicroNIRConfig(CompareConfig):
    BASE_DIM = 64
    NUM_BLOCKS = 6
    EXPANSION_RATIO = 2
    SE_RATIO = 4
    DROPOUT = 0.05
    DROP_PATH_RATE = 0.1
    ACTIVATION = 'gelu'
    OUTPUT_CONSTRAINT = 'tanh'

print("Model configs defined")

## 5. Load Test Dataset

In [ ]:
def parse_tfrecord(example, config):
    """Parse TFRecord with validity mask."""
    feature_description = {
        'B2': tf.io.VarLenFeature(tf.float32),
        'B3': tf.io.VarLenFeature(tf.float32),
        'B4': tf.io.VarLenFeature(tf.float32),
        'B5': tf.io.VarLenFeature(tf.float32),
        'B6': tf.io.VarLenFeature(tf.float32),
        'B7': tf.io.VarLenFeature(tf.float32),
        'B8': tf.io.VarLenFeature(tf.float32),
        'valid': tf.io.VarLenFeature(tf.float32),
    }
    parsed = tf.io.parse_single_example(example, feature_description)

    def find_dimensions(num_values):
        num_values_f = tf.cast(num_values, tf.float32)
        sqrt_n = tf.sqrt(num_values_f)
        sqrt_n_int = tf.cast(sqrt_n, tf.int32)
        is_square = tf.equal(sqrt_n_int * sqrt_n_int, num_values)
        def get_square_dims(): return sqrt_n_int, sqrt_n_int
        def find_divisor():
            candidates = tf.constant([64, 65, 66, 128, 129, 130, 256], dtype=tf.int32)
            remainders = num_values % candidates
            valid_mask = tf.equal(remainders, 0)
            first_valid_idx = tf.argmax(tf.cast(valid_mask, tf.int32))
            height = candidates[first_valid_idx]
            width = num_values // height
            any_valid = tf.reduce_any(valid_mask)
            height = tf.cond(any_valid, lambda: height, lambda: sqrt_n_int + 1)
            width = tf.cond(any_valid, lambda: width, lambda: num_values // (sqrt_n_int + 1))
            return height, width
        return tf.cond(is_square, get_square_dims, find_divisor)

    def extract_band(band_name):
        dense = tf.sparse.to_dense(parsed[band_name])
        dense = tf.where(tf.math.is_nan(dense), tf.zeros_like(dense), dense)
        num_values = tf.shape(dense)[0]
        height, width = find_dimensions(num_values)
        return tf.reshape(dense, [height, width])

    b2, b3, b4 = extract_band('B2'), extract_band('B3'), extract_band('B4')
    rgb = tf.stack([b4, b3, b2], axis=-1)

    b5, b6, b7, b8 = extract_band('B5'), extract_band('B6'), extract_band('B7'), extract_band('B8')
    nir_rededge = tf.stack([b5, b6, b7, b8], axis=-1)

    valid_mask = extract_band('valid')

    rgb = tf.clip_by_value(rgb, 0.0, 10000.0) / 10000.0
    nir_rededge = tf.clip_by_value(nir_rededge, 0.0, 10000.0) / 10000.0

    rgb_sum = tf.reduce_sum(rgb, axis=-1)
    nir_sum = tf.reduce_sum(nir_rededge, axis=-1)
    non_black = tf.cast((rgb_sum > 0.01) | (nir_sum > 0.01), tf.float32)
    mask = valid_mask * non_black

    return tf.cast(rgb, tf.float32), tf.cast(nir_rededge, tf.float32), tf.cast(mask, tf.float32)


def center_crop(rgb, nir, mask, crop_size):
    h, w = tf.shape(rgb)[0], tf.shape(rgb)[1]
    offset_h = tf.maximum((h - crop_size) // 2, 0)
    offset_w = tf.maximum((w - crop_size) // 2, 0)

    rgb = rgb[offset_h:offset_h + crop_size, offset_w:offset_w + crop_size, :]
    nir = nir[offset_h:offset_h + crop_size, offset_w:offset_w + crop_size, :]
    mask = mask[offset_h:offset_h + crop_size, offset_w:offset_w + crop_size]

    rgb = tf.image.resize_with_crop_or_pad(rgb, crop_size, crop_size)
    nir = tf.image.resize_with_crop_or_pad(nir, crop_size, crop_size)
    mask = tf.image.resize_with_crop_or_pad(mask[..., tf.newaxis], crop_size, crop_size)[..., 0]

    return rgb, nir, mask


def load_test_dataset(config):
    pattern = f"gs://{config.BUCKET}/{config.FOLDER}/test/*.tfrecord.gz"
    files = tf.io.gfile.glob(pattern)
    print(f"Found {len(files)} test files")

    dataset = tf.data.TFRecordDataset(files, compression_type='GZIP')
    dataset = dataset.map(lambda x: parse_tfrecord(x, config), num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.map(lambda r, n, m: center_crop(r, n, m, config.PATCH_SIZE), num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(config.BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

    return dataset

test_ds = load_test_dataset(config)
print("Test dataset loaded")

## 6. Evaluation Metrics (Remote Sensing 2025 Paper)

In [ ]:
def compute_metrics(model, dataset, config, model_name="Model"):
    """
    Compute all metrics aligned with Remote Sensing 2025 paper:
    - MRAE (Mean Relative Absolute Error)
    - RMSE (Root Mean Square Error)
    - MAE (Mean Absolute Error)
    - SAM (Spectral Angle Mapper)
    - SID (Spectral Information Divergence)
    - NDVI MAE / NDRE MAE
    """
    eps = 1e-7

    # Accumulators (float64 for precision)
    mae_sum = tf.zeros([4], dtype=tf.float64)
    mse_sum = tf.zeros([4], dtype=tf.float64)
    mrae_sum = tf.zeros([4], dtype=tf.float64)
    sam_sum = tf.constant(0.0, dtype=tf.float64)
    sid_sum = tf.constant(0.0, dtype=tf.float64)
    ndvi_sum = tf.constant(0.0, dtype=tf.float64)
    ndre_sum = tf.constant(0.0, dtype=tf.float64)
    valid_sum = tf.constant(0.0, dtype=tf.float64)
    ndvi_valid = tf.constant(0.0, dtype=tf.float64)
    ndre_valid = tf.constant(0.0, dtype=tf.float64)

    n_batches = 0
    for rgb, nir, mask in dataset:
        pred = model(rgb, training=False)

        # Ensure float32
        pred = tf.cast(pred, tf.float32)
        nir = tf.cast(nir, tf.float32)
        mask_f = tf.cast(mask, tf.float32)
        mask_expanded = mask_f[..., tf.newaxis]

        valid_pixels = tf.reduce_sum(mask_f)
        if valid_pixels < 1:
            continue

        # MAE per band
        abs_diff = tf.abs(nir - pred) * mask_expanded
        mae_sum += tf.cast(tf.reduce_sum(abs_diff, axis=[0, 1, 2]), tf.float64)

        # MSE per band
        sq_diff = tf.square(nir - pred) * mask_expanded
        mse_sum += tf.cast(tf.reduce_sum(sq_diff, axis=[0, 1, 2]), tf.float64)

        # MRAE per band
        rel_abs_error = tf.abs(nir - pred) / (tf.abs(nir) + eps) * mask_expanded
        mrae_sum += tf.cast(tf.reduce_sum(rel_abs_error, axis=[0, 1, 2]), tf.float64)

        # SAM (Spectral Angle Mapper)
        norm_true = tf.sqrt(tf.reduce_sum(tf.square(nir), axis=-1) + eps)
        norm_pred = tf.sqrt(tf.reduce_sum(tf.square(pred), axis=-1) + eps)
        dot = tf.reduce_sum(nir * pred, axis=-1) / (norm_true * norm_pred + eps)
        dot = tf.clip_by_value(dot, -1.0 + eps, 1.0 - eps)
        angle = tf.acos(dot)
        sam_sum += tf.cast(tf.reduce_sum(angle * mask_f), tf.float64)

        # SID (Spectral Information Divergence)
        pred_clip = tf.clip_by_value(pred, config.OUTPUT_MIN, config.OUTPUT_MAX)
        nir_clip = tf.clip_by_value(nir, config.OUTPUT_MIN, config.OUTPUT_MAX)
        x = tf.maximum(nir_clip, 0.0) + eps
        y = tf.maximum(pred_clip, 0.0) + eps
        x_sum = tf.reduce_sum(x, axis=-1, keepdims=True)
        y_sum = tf.reduce_sum(y, axis=-1, keepdims=True)
        p = x / (x_sum + eps)
        q = y / (y_sum + eps)
        sid_map = tf.reduce_sum(p * tf.math.log(p / q), axis=-1) + tf.reduce_sum(q * tf.math.log(q / p), axis=-1)
        sid_sum += tf.cast(tf.reduce_sum(sid_map * mask_f), tf.float64)

        # NDVI / NDRE
        red = tf.clip_by_value(rgb[..., 0], 0.0, 1.0)
        nir_b8_true = nir_clip[..., 3]
        nir_b8_pred = pred_clip[..., 3]

        den_ndvi = nir_b8_true + red
        ndvi_mask = tf.cast(den_ndvi > 0.05, tf.float32)
        ndvi_true = (nir_b8_true - red) / (den_ndvi + eps)
        ndvi_pred = (nir_b8_pred - red) / (nir_b8_pred + red + eps)
        ndvi_sum += tf.cast(tf.reduce_sum(tf.abs(ndvi_true - ndvi_pred) * mask_f * ndvi_mask), tf.float64)
        ndvi_valid += tf.cast(tf.reduce_sum(mask_f * ndvi_mask), tf.float64)

        re_true = nir_clip[..., 0]  # B5
        re_pred = pred_clip[..., 0]
        den_ndre = nir_b8_true + re_true
        ndre_mask = tf.cast(den_ndre > 0.05, tf.float32)
        ndre_true = (nir_b8_true - re_true) / (den_ndre + eps)
        ndre_pred = (nir_b8_pred - re_pred) / (nir_b8_pred + re_pred + eps)
        ndre_sum += tf.cast(tf.reduce_sum(tf.abs(ndre_true - ndre_pred) * mask_f * ndre_mask), tf.float64)
        ndre_valid += tf.cast(tf.reduce_sum(mask_f * ndre_mask), tf.float64)

        valid_sum += tf.cast(valid_pixels, tf.float64)
        n_batches += 1

    # Aggregate
    valid_sum = tf.maximum(valid_sum, tf.constant(1.0, dtype=tf.float64))

    mae_per_band = mae_sum / valid_sum
    rmse_per_band = tf.sqrt(mse_sum / valid_sum)
    mrae_per_band = mrae_sum / valid_sum

    overall_mae = tf.reduce_sum(mae_sum) / (valid_sum * 4.0)
    overall_rmse = tf.sqrt(tf.reduce_sum(mse_sum) / (valid_sum * 4.0))
    overall_mrae = tf.reduce_sum(mrae_sum) / (valid_sum * 4.0)
    sam_mean = sam_sum / valid_sum
    sid_mean = sid_sum / valid_sum

    ndvi_valid = tf.maximum(ndvi_valid, tf.constant(1.0, dtype=tf.float64))
    ndre_valid = tf.maximum(ndre_valid, tf.constant(1.0, dtype=tf.float64))
    ndvi_mae = ndvi_sum / ndvi_valid
    ndre_mae = ndre_sum / ndre_valid

    # Paper-style grouping
    rededge_mrae = tf.reduce_mean(tf.gather(mrae_per_band, [0, 1, 2]))
    nir_mrae = mrae_per_band[3]

    results = {
        'model': model_name,
        'mae': float(overall_mae),
        'rmse': float(overall_rmse),
        'mrae': float(overall_mrae),
        'sam': float(sam_mean),
        'sid': float(sid_mean),
        'ndvi_mae': float(ndvi_mae),
        'ndre_mae': float(ndre_mae),
        'rededge_mrae': float(rededge_mrae),
        'nir_mrae': float(nir_mrae),
        'mae_per_band': [float(x) for x in mae_per_band.numpy()],
        'rmse_per_band': [float(x) for x in rmse_per_band.numpy()],
        'mrae_per_band': [float(x) for x in mrae_per_band.numpy()],
    }

    return results

print("Metrics function defined")

## 7. Run Evaluation

In [ ]:
import gc

# =================================================================
# Sequential model evaluation (one at a time to save memory)
# =================================================================

# Prepare visualization samples first (cache a few sample tensors)
print("Preparing visualization samples...")
viz_ds = test_ds.unbatch().shuffle(buffer_size=1000).batch(4)
viz_rgb, viz_nir, viz_mask = None, None, None
for batch in viz_ds.take(1):
    viz_rgb = batch[0].numpy()
    viz_nir = batch[1].numpy()
    viz_mask = batch[2].numpy()
print(f"Cached {viz_rgb.shape[0]} samples for visualization ({viz_rgb.shape})")

# Model registry: name -> (builder, config, weights_path)
model_registry = {
    'WaveletMST': (build_wavelet_mst, WaveletConfig(), config.WAVELET_WEIGHTS),
    'CNN_MSGPP': (build_cnn_msgpp_decoder, CNNMSGPPConfig(), config.CNN_MSGPP_WEIGHTS),
    'MSTPP': (build_mstpp, MSTPPConfig(), config.MSTPP_WEIGHTS),
    'MicroNIR': (build_micronir, MicroNIRConfig(), config.MICRONIR_WEIGHTS),
}

all_results = {}
all_predictions = {}  # name -> numpy array of predictions on viz samples

for name, (builder, model_config, weights_path) in model_registry.items():
    print(f"\n{'='*60}")
    print(f"Loading & evaluating: {name}")
    print(f"{'='*60}")

    try:
        # Build model
        model = builder(model_config)

        # Load weights
        if os.path.exists(weights_path):
            model.load_weights(weights_path)
            print(f"  Weights loaded - {model.count_params():,} params")
        else:
            print(f"  WARNING: Weights not found: {weights_path}")

        # Compute metrics on full test set
        results = compute_metrics(model, test_ds, config, model_name=name)
        all_results[name] = results

        print(f"  MAE:  {results['mae']:.4f}")
        print(f"  RMSE: {results['rmse']:.4f}")
        print(f"  MRAE: {results['mrae']:.4f}")
        print(f"  SAM:  {results['sam']:.4f} rad")
        print(f"  SID:  {results['sid']:.4f}")
        print(f"  NDVI MAE: {results['ndvi_mae']:.4f}")
        print(f"  NDRE MAE: {results['ndre_mae']:.4f}")

        # Cache predictions on viz samples (small numpy arrays, not the model)
        all_predictions[name] = model(tf.constant(viz_rgb), training=False).numpy()

    except Exception as e:
        print(f"  Error: {e}")
        import traceback
        traceback.print_exc()

    finally:
        # Free model memory before loading the next one
        try:
            del model
        except:
            pass
        tf.keras.backend.clear_session()
        gc.collect()
        print(f"  Memory freed for {name}")

print(f"\n{'='*60}")
print(f"Done! Evaluated {len(all_results)} models: {list(all_results.keys())}")
print(f"Visualization predictions cached: {list(all_predictions.keys())}")

In [ ]:
# Display results summary (metrics already computed above)
print("=" * 60)
print("EVALUATION SUMMARY")
print("=" * 60)

for name, results in all_results.items():
    print(f"\n{name}:")
    print(f"  MAE:  {results['mae']:.4f}  |  RMSE: {results['rmse']:.4f}  |  MRAE: {results['mrae']:.4f}")
    print(f"  SAM:  {results['sam']:.4f}  |  SID:  {results['sid']:.4f}")
    print(f"  NDVI MAE: {results['ndvi_mae']:.4f}  |  NDRE MAE: {results['ndre_mae']:.4f}")

print(f"\nAll metrics available in 'all_results' dict.")

## 8. Results Table

In [ ]:
import pandas as pd

# Create comparison table
table_data = []
for name, results in all_results.items():
    table_data.append({
        'Model': name,
        'MAE': f"{results['mae']:.4f}",
        'RMSE': f"{results['rmse']:.4f}",
        'MRAE': f"{results['mrae']:.4f}",
        'SAM (rad)': f"{results['sam']:.4f}",
        'SID': f"{results['sid']:.4f}",
        'NDVI MAE': f"{results['ndvi_mae']:.4f}",
        'NDRE MAE': f"{results['ndre_mae']:.4f}",
        'RedEdge MRAE': f"{results['rededge_mrae']:.4f}",
        'NIR MRAE': f"{results['nir_mrae']:.4f}",
    })

df = pd.DataFrame(table_data)
print("\n" + "="*80)
print("MODEL COMPARISON - RGB to NIR/RedEdge Reconstruction")
print("="*80)
print(df.to_string(index=False))
print("="*80)

# Save to CSV
df.to_csv('model_comparison_results.csv', index=False)
print("\nResults saved to model_comparison_results.csv")

In [ ]:
# Per-band comparison table
band_names = ['B5', 'B6', 'B7', 'B8']

print("\n" + "="*60)
print("MAE PER BAND")
print("="*60)
band_table = []
for name, results in all_results.items():
    row = {'Model': name}
    for i, band in enumerate(band_names):
        row[band] = f"{results['mae_per_band'][i]:.4f}"
    band_table.append(row)

df_bands = pd.DataFrame(band_table)
print(df_bands.to_string(index=False))

print("\n" + "="*60)
print("MRAE PER BAND (Paper metric)")
print("="*60)
mrae_table = []
for name, results in all_results.items():
    row = {'Model': name}
    for i, band in enumerate(band_names):
        row[band] = f"{results['mrae_per_band'][i]:.4f}"
    mrae_table.append(row)

df_mrae = pd.DataFrame(mrae_table)
print(df_mrae.to_string(index=False))

## 9. Visual Comparison

### 9.1 Per-Band Reconstruction (B5, B6, B7, B8)

Grid layout: **rows = spectral bands**, **columns = Ground Truth + models**.
Below, an error map grid using the `viridis` colormap (perceptually uniform, most used in scientific literature).

In [ ]:
import os

Upload_drive = True  # Mettre à False pour enregistrer en local

# Définition du chemin selon l'option choisie
if Upload_drive:
    save_dir = '/content/drive/MyDrive/images/'  # Modifie ce chemin selon ton arborescence Drive
else:
    save_dir = 'images/'

def compute_ndvi(nir_b8, red, eps=1e-7):
    """Compute NDVI = (NIR - Red) / (NIR + Red)"""
    return (nir_b8 - red) / (nir_b8 + red + eps)

def compute_ndre(nir_b8, rededge_b5, eps=1e-7):
    """Compute NDRE = (NIR - RedEdge) / (NIR + RedEdge)"""
    return (nir_b8 - rededge_b5) / (nir_b8 + rededge_b5 + eps)


# ─── Common style settings ───────────────────────────────────────────────
BAND_NAMES = ['B5 (RE1)', 'B6 (RE2)', 'B7 (RE3)', 'B8 (NIR)']
BAND_VMAX  = [0.50, 0.50, 0.50, 0.60]          # display range per band
ERROR_VMAX = [0.08, 0.08, 0.08, 0.10]           # error range per band
PRED_CMAP  = 'RdYlGn'                           # reflectance colormap
ERR_CMAP   = 'viridis'                          # error colormap (perceptually uniform)


def visualize_band_grid(predictions_dict, rgb_data, nir_data, mask_data,
                        sample_idx=0):
    """
    Two publication-ready grids for ONE sample:
      • Grid 1 – Reconstruction: rows=bands, cols=[GT, model1, model2, …]
      • Grid 2 – Absolute Error:  same layout, viridis colormap
    An RGB thumbnail is shown as a row-spanning inset on the left.
    """
    model_names = list(predictions_dict.keys())
    n_models = len(model_names)
    n_bands  = 4                            # B5 B6 B7 B8
    n_cols   = 1 + n_models                 # GT + models

    rgb   = rgb_data[sample_idx]
    gt    = nir_data[sample_idx]
    mask  = mask_data[sample_idx]

    # S'assurer que le dossier de sauvegarde existe
    os.makedirs(save_dir, exist_ok=True)

    # ── Grid 1: Reconstruction ──────────────────────────────────────────
    fig, axes = plt.subplots(n_bands, n_cols, figsize=(3.2 * n_cols, 3.0 * n_bands))
    fig.suptitle(f'Per-Band Reconstruction — Sample {sample_idx + 1}',
                 fontsize=15, fontweight='bold', y=1.01)

    for b in range(n_bands):
        # Ground Truth column
        im = axes[b, 0].imshow(gt[:, :, b], cmap=PRED_CMAP,
                                vmin=0, vmax=BAND_VMAX[b])
        axes[b, 0].set_ylabel(BAND_NAMES[b], fontsize=12, fontweight='bold',
                               rotation=0, labelpad=60, va='center')
        if b == 0:
            axes[b, 0].set_title('Ground Truth', fontsize=11, fontweight='bold')
        axes[b, 0].axis('off')
        # keep ylabel visible
        axes[b, 0].yaxis.set_visible(True)
        axes[b, 0].tick_params(left=False, labelleft=False)

        # Model columns
        for j, name in enumerate(model_names):
            pred = predictions_dict[name][sample_idx]
            axes[b, 1 + j].imshow(np.clip(pred[:, :, b], 0, 1),
                                   cmap=PRED_CMAP, vmin=0, vmax=BAND_VMAX[b])
            if b == 0:
                axes[b, 1 + j].set_title(name, fontsize=11, fontweight='bold')
            axes[b, 1 + j].axis('off')

    # Shared colorbar
    cbar = fig.colorbar(im, ax=axes[:, -1].tolist(), fraction=0.025, pad=0.02)
    cbar.set_label('Reflectance', fontsize=10)

    plt.tight_layout()

    # Sauvegarde Grid 1
    file_name_recon = f'band_reconstruction_sample_{sample_idx + 1}.png'
    save_path_recon = os.path.join(save_dir, file_name_recon)
    plt.savefig(save_path_recon, dpi=150, bbox_inches='tight')
    print(f"Image sauvegardée : {save_path_recon}")

    plt.show()

    # ── Grid 2: Absolute Error ──────────────────────────────────────────
    fig2, axes2 = plt.subplots(n_bands, n_models, figsize=(3.2 * n_models, 3.0 * n_bands))
    fig2.suptitle(f'Absolute Error |GT − Pred| — Sample {sample_idx + 1}',
                  fontsize=15, fontweight='bold', y=1.01)

    if n_models == 1:
        axes2 = axes2[:, np.newaxis]       # ensure 2-D indexing

    for b in range(n_bands):
        for j, name in enumerate(model_names):
            pred = predictions_dict[name][sample_idx]
            error = np.abs(gt[:, :, b] - np.clip(pred[:, :, b], 0, 1)) * mask
            mae = np.sum(error) / (np.sum(mask) + 1e-8)

            im2 = axes2[b, j].imshow(error, cmap=ERR_CMAP,
                                      vmin=0, vmax=ERROR_VMAX[b])
            if b == 0:
                axes2[b, j].set_title(name, fontsize=11, fontweight='bold')

            # MAE as small annotation inside the image
            axes2[b, j].text(0.98, 0.02, f'MAE {mae:.4f}',
                             transform=axes2[b, j].transAxes, fontsize=9,
                             color='white', ha='right', va='bottom',
                             bbox=dict(facecolor='black', alpha=0.6, pad=2,
                                       boxstyle='round,pad=0.3'))
            axes2[b, j].axis('off')

        # Band label on first column
        axes2[b, 0].set_ylabel(BAND_NAMES[b], fontsize=12, fontweight='bold',
                                rotation=0, labelpad=60, va='center')
        axes2[b, 0].yaxis.set_visible(True)
        axes2[b, 0].tick_params(left=False, labelleft=False)

    cbar2 = fig2.colorbar(im2, ax=axes2[:, -1].tolist(), fraction=0.025, pad=0.02)
    cbar2.set_label('Absolute Error', fontsize=10)

    plt.tight_layout()

    # Sauvegarde Grid 2
    file_name_err = f'band_error_sample_{sample_idx + 1}.png'
    save_path_err = os.path.join(save_dir, file_name_err)
    plt.savefig(save_path_err, dpi=150, bbox_inches='tight')
    print(f"Image sauvegardée : {save_path_err}")

    plt.show()

print("Band-grid visualization functions defined")

In [ ]:
# Generate per-band reconstruction + error grids
for s in range(min(4, viz_rgb.shape[0])):
    visualize_band_grid(all_predictions, viz_rgb, viz_nir, viz_mask, sample_idx=s)

### 9.2 False-Color Composite (NIR-R-G) Comparison

False-color composite using **B8 (NIR) – B4 (Red) – B3 (Green)** channels, commonly used to highlight vegetation (appears bright red).

In [ ]:
def visualize_false_color(predictions_dict, rgb_data, nir_data, mask_data,
                          num_samples=4):
    """
    False-color composite (B8-Red-Green) comparison.
    Single row: RGB input | GT false-color | model false-colors …
    """
    model_names = list(predictions_dict.keys())
    n_cols = 2 + len(model_names)

    for i in range(min(num_samples, rgb_data.shape[0])):
        rgb  = rgb_data[i]
        gt   = nir_data[i]

        fig, axes = plt.subplots(1, n_cols, figsize=(3.5 * n_cols, 3.5))
        fig.suptitle(f'False-Color Composite (NIR-R-G) — Sample {i + 1}',
                     fontsize=13, fontweight='bold')

        # RGB input
        axes[0].imshow(np.clip(rgb * 3, 0, 1))
        axes[0].set_title('RGB Input', fontsize=10, fontweight='bold')
        axes[0].axis('off')

        # GT false-color: B8, Red(B4=rgb[:,,:,0]), Green(B3=rgb[:,:,1])
        gt_fc = np.stack([np.clip(gt[:, :, 3], 0, 1),
                          np.clip(rgb[:, :, 0], 0, 1),
                          np.clip(rgb[:, :, 1], 0, 1)], axis=-1)
        axes[1].imshow(np.clip(gt_fc * 2.5, 0, 1))
        axes[1].set_title('Ground Truth', fontsize=10, fontweight='bold')
        axes[1].axis('off')

        for j, name in enumerate(model_names):
            pred = predictions_dict[name][i]
            pred_fc = np.stack([np.clip(pred[:, :, 3], 0, 1),
                                np.clip(rgb[:, :, 0], 0, 1),
                                np.clip(rgb[:, :, 1], 0, 1)], axis=-1)
            axes[2 + j].imshow(np.clip(pred_fc * 2.5, 0, 1))
            axes[2 + j].set_title(name, fontsize=10, fontweight='bold')
            axes[2 + j].axis('off')

        plt.tight_layout()
        plt.show()

print("False-color visualization defined")

In [ ]:
visualize_false_color(all_predictions, viz_rgb, viz_nir, viz_mask, num_samples=4)

### 9.3 NDVI & NDRE Index Comparison
- **Row 1** — NDVI maps (GT + models)
- **Row 2** — NDRE maps (GT + models)


In [ ]:
def visualize_indices_grid(predictions_dict, rgb_data, nir_data, mask_data,
                           sample_idx=0):
    """
    Two grids per sample:
      Grid 1 – Index values : rows=[NDVI, NDRE], cols=[GT, model1, …]
      Grid 2 – Index errors : rows=[NDVI, NDRE], cols=[model1, …] (viridis)
    """
    model_names = list(predictions_dict.keys())
    n_models = len(model_names)
    eps = 1e-7

    rgb  = rgb_data[sample_idx]
    gt   = nir_data[sample_idx]
    mask = mask_data[sample_idx]

    red       = np.clip(rgb[:, :, 0], 0, 1)
    gt_b8     = np.clip(gt[:, :, 3], 0, 1)
    gt_b5     = np.clip(gt[:, :, 0], 0, 1)
    ndvi_true = np.clip((gt_b8 - red)  / (gt_b8 + red  + eps), -1, 1)
    ndre_true = np.clip((gt_b8 - gt_b5) / (gt_b8 + gt_b5 + eps), -1, 1)

    index_info = [
        ('NDVI', ndvi_true, -0.2, 0.8, 0.20),
        ('NDRE', ndre_true, -0.1, 0.5, 0.15),
    ]

    n_cols_val = 1 + n_models   # GT + models

    # ── Grid 1: Index Values ────────────────────────────────────────────
    fig, axes = plt.subplots(2, n_cols_val, figsize=(3.3 * n_cols_val, 6.5))
    fig.suptitle(f'Vegetation Index Comparison — Sample {sample_idx + 1}',
                 fontsize=15, fontweight='bold', y=1.01)

    for r, (idx_name, gt_idx, vmin, vmax, _) in enumerate(index_info):
        # GT column
        im = axes[r, 0].imshow(gt_idx, cmap='RdYlGn', vmin=vmin, vmax=vmax)
        axes[r, 0].set_ylabel(idx_name, fontsize=13, fontweight='bold',
                               rotation=0, labelpad=50, va='center')
        if r == 0:
            axes[r, 0].set_title('Ground Truth', fontsize=11, fontweight='bold')
        axes[r, 0].axis('off')
        axes[r, 0].yaxis.set_visible(True)
        axes[r, 0].tick_params(left=False, labelleft=False)

        # Model columns
        for j, name in enumerate(model_names):
            pred = predictions_dict[name][sample_idx]
            p_b8 = np.clip(pred[:, :, 3], 0, 1)
            p_b5 = np.clip(pred[:, :, 0], 0, 1)
            if idx_name == 'NDVI':
                pred_idx = np.clip((p_b8 - red) / (p_b8 + red + eps), -1, 1)
            else:
                pred_idx = np.clip((p_b8 - p_b5) / (p_b8 + p_b5 + eps), -1, 1)

            axes[r, 1 + j].imshow(pred_idx, cmap='RdYlGn', vmin=vmin, vmax=vmax)
            if r == 0:
                axes[r, 1 + j].set_title(name, fontsize=11, fontweight='bold')
            axes[r, 1 + j].axis('off')

    cbar = fig.colorbar(im, ax=axes[:, -1].tolist(), fraction=0.025, pad=0.02)
    cbar.set_label('Index Value', fontsize=10)
    plt.tight_layout()
    plt.savefig(f'indices_values_sample_{sample_idx + 1}.png',
                dpi=150, bbox_inches='tight')
    plt.show()

    # ── Grid 2: Index Errors ────────────────────────────────────────────
    fig2, axes2 = plt.subplots(2, n_models, figsize=(3.3 * n_models, 6.5))
    fig2.suptitle(f'Index Absolute Error — Sample {sample_idx + 1}',
                  fontsize=15, fontweight='bold', y=1.01)

    if n_models == 1:
        axes2 = axes2[:, np.newaxis]

    for r, (idx_name, gt_idx, _, _, evmax) in enumerate(index_info):
        for j, name in enumerate(model_names):
            pred = predictions_dict[name][sample_idx]
            p_b8 = np.clip(pred[:, :, 3], 0, 1)
            p_b5 = np.clip(pred[:, :, 0], 0, 1)
            if idx_name == 'NDVI':
                pred_idx = np.clip((p_b8 - red) / (p_b8 + red + eps), -1, 1)
            else:
                pred_idx = np.clip((p_b8 - p_b5) / (p_b8 + p_b5 + eps), -1, 1)

            error = np.abs(gt_idx - pred_idx) * mask
            mae   = np.sum(error) / (np.sum(mask) + 1e-8)

            im2 = axes2[r, j].imshow(error, cmap=ERR_CMAP, vmin=0, vmax=evmax)
            if r == 0:
                axes2[r, j].set_title(name, fontsize=11, fontweight='bold')
            axes2[r, j].text(0.98, 0.02, f'MAE {mae:.4f}',
                             transform=axes2[r, j].transAxes, fontsize=9,
                             color='white', ha='right', va='bottom',
                             bbox=dict(facecolor='black', alpha=0.6, pad=2,
                                       boxstyle='round,pad=0.3'))
            axes2[r, j].axis('off')

        axes2[r, 0].set_ylabel(idx_name, fontsize=13, fontweight='bold',
                                rotation=0, labelpad=50, va='center')
        axes2[r, 0].yaxis.set_visible(True)
        axes2[r, 0].tick_params(left=False, labelleft=False)

    cbar2 = fig2.colorbar(im2, ax=axes2[:, -1].tolist(), fraction=0.025, pad=0.02)
    cbar2.set_label('Absolute Error', fontsize=10)
    plt.tight_layout()
    plt.savefig(f'indices_error_sample_{sample_idx + 1}.png',
                dpi=150, bbox_inches='tight')
    plt.show()

print("NDVI/NDRE grid visualization defined")

In [ ]:
for s in range(min(4, viz_rgb.shape[0])):
    visualize_indices_grid(all_predictions, viz_rgb, viz_nir, viz_mask, sample_idx=s)

## 10. Bar Chart Comparison

In [ ]:
def plot_metrics_comparison(all_results):
    """Create bar charts comparing all models on key metrics."""
    model_names = list(all_results.keys())
    n_models = len(model_names)

    metrics = ['mae', 'rmse', 'mrae', 'sam', 'sid', 'ndvi_mae', 'ndre_mae']
    metric_labels = ['MAE', 'RMSE', 'MRAE', 'SAM (rad)', 'SID', 'NDVI MAE', 'NDRE MAE']

    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()

    colors = plt.cm.Set2(np.linspace(0, 1, n_models))

    for idx, (metric, label) in enumerate(zip(metrics, metric_labels)):
        values = [all_results[name][metric] for name in model_names]

        bars = axes[idx].bar(model_names, values, color=colors)
        axes[idx].set_title(label, fontsize=12, fontweight='bold')
        axes[idx].set_ylabel(label)
        axes[idx].tick_params(axis='x', rotation=45)

        # Add value labels
        for bar, val in zip(bars, values):
            axes[idx].text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                          f'{val:.4f}', ha='center', va='bottom', fontsize=9)

        # Highlight best (lowest)
        best_idx = np.argmin(values)
        bars[best_idx].set_edgecolor('green')
        bars[best_idx].set_linewidth(3)

    # Hide last subplot
    axes[-1].axis('off')

    plt.suptitle('Model Comparison - Key Metrics (lower is better)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('model_comparison_chart.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Chart saved to model_comparison_chart.png")

plot_metrics_comparison(all_results)

In [ ]:
def plot_band_comparison(all_results):
    """Create bar charts comparing MRAE per band."""
    model_names = list(all_results.keys())
    band_names = ['B5', 'B6', 'B7', 'B8']

    x = np.arange(len(band_names))
    width = 0.2

    fig, ax = plt.subplots(figsize=(12, 6))
    colors = plt.cm.Set2(np.linspace(0, 1, len(model_names)))

    for i, name in enumerate(model_names):
        values = all_results[name]['mrae_per_band']
        bars = ax.bar(x + i*width, values, width, label=name, color=colors[i])

    ax.set_xlabel('Band', fontsize=12)
    ax.set_ylabel('MRAE', fontsize=12)
    ax.set_title('MRAE per Band - Model Comparison', fontsize=14, fontweight='bold')
    ax.set_xticks(x + width * (len(model_names)-1) / 2)
    ax.set_xticklabels(band_names)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig('band_comparison_chart.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Chart saved to band_comparison_chart.png")

plot_band_comparison(all_results)

## 11. Summary

In [ ]:
print("\n" + "="*80)
print("SUMMARY - BEST MODEL PER METRIC")
print("="*80)

metrics_to_check = ['mae', 'rmse', 'mrae', 'sam', 'sid', 'ndvi_mae', 'ndre_mae']
metric_labels = ['MAE', 'RMSE', 'MRAE', 'SAM', 'SID', 'NDVI MAE', 'NDRE MAE']

summary = []
for metric, label in zip(metrics_to_check, metric_labels):
    best_model = min(all_results.keys(), key=lambda x: all_results[x][metric])
    best_value = all_results[best_model][metric]
    summary.append({'Metric': label, 'Best Model': best_model, 'Value': f"{best_value:.4f}"})
    print(f"{label:12s}: {best_model:20s} ({best_value:.4f})")

print("="*80)

# Count wins
win_counts = {}
for item in summary:
    model = item['Best Model']
    win_counts[model] = win_counts.get(model, 0) + 1

print("\nWINS PER MODEL:")
for model, wins in sorted(win_counts.items(), key=lambda x: -x[1]):
    print(f"  {model}: {wins}/{len(metrics_to_check)} metrics")

overall_best = max(win_counts.keys(), key=lambda x: win_counts[x])
print(f"\nOVERALL BEST: {overall_best}")

## 12. Publication Figure — Complete Comparison

Single figure per sample: **rows = [B5, B6, B7, B8, NDVI, NDRE]**, **columns = [Ground Truth, models …]**, plus a separate error grid underneath.

In [ ]:
def visualize_complete_comparison(predictions_dict, rgb_data, nir_data, mask_data,
                                  num_samples=2):
    """
    Publication-quality figure per sample.

    Grid 1 – Values:
        rows  = B5, B6, B7, B8, NDVI, NDRE   (6 rows)
        cols  = Ground Truth, Model1, Model2, … (1 + N cols)

    Grid 2 – Absolute Error (viridis):
        rows  = same 6
        cols  = Model1, Model2, …              (N cols)
    """
    model_names = list(predictions_dict.keys())
    n_models = len(model_names)
    eps = 1e-7

    row_labels = ['B5 (RE1)', 'B6 (RE2)', 'B7 (RE3)', 'B8 (NIR)', 'NDVI', 'NDRE']
    n_rows = len(row_labels)

    for i in range(min(num_samples, rgb_data.shape[0])):
        rgb  = rgb_data[i]
        gt   = nir_data[i]
        mask = mask_data[i]

        red   = np.clip(rgb[:, :, 0], 0, 1)
        gt_b8 = np.clip(gt[:, :, 3], 0, 1)
        gt_b5 = np.clip(gt[:, :, 0], 0, 1)
        ndvi_true = np.clip((gt_b8 - red)  / (gt_b8 + red  + eps), -1, 1)
        ndre_true = np.clip((gt_b8 - gt_b5) / (gt_b8 + gt_b5 + eps), -1, 1)

        # Build data arrays: 6 ground-truth maps
        gt_maps = [gt[:, :, 0], gt[:, :, 1], gt[:, :, 2], gt[:, :, 3],
                   ndvi_true, ndre_true]
        vmins = [0, 0, 0, 0, -0.2, -0.1]
        vmaxs = [0.50, 0.50, 0.50, 0.60, 0.80, 0.50]
        evmaxs = [0.08, 0.08, 0.08, 0.10, 0.20, 0.15]

        # ── Grid 1: Values ──────────────────────────────────────────────
        n_cols_val = 1 + n_models
        fig, axes = plt.subplots(n_rows, n_cols_val,
                                  figsize=(3.0 * n_cols_val, 2.6 * n_rows))
        fig.suptitle(f'Complete Comparison — Sample {i + 1}',
                     fontsize=16, fontweight='bold', y=1.005)

        for r in range(n_rows):
            # GT
            axes[r, 0].imshow(gt_maps[r], cmap='RdYlGn',
                               vmin=vmins[r], vmax=vmaxs[r])
            if r == 0:
                axes[r, 0].set_title('Ground Truth', fontsize=10, fontweight='bold')
            axes[r, 0].set_ylabel(row_labels[r], fontsize=11, fontweight='bold',
                                   rotation=0, labelpad=60, va='center')
            axes[r, 0].axis('off')
            axes[r, 0].yaxis.set_visible(True)
            axes[r, 0].tick_params(left=False, labelleft=False)

            # Models
            for j, name in enumerate(model_names):
                pred = predictions_dict[name][i]
                p_b8 = np.clip(pred[:, :, 3], 0, 1)
                p_b5 = np.clip(pred[:, :, 0], 0, 1)

                if r < 4:
                    data = np.clip(pred[:, :, r], 0, 1)
                elif r == 4:
                    data = np.clip((p_b8 - red) / (p_b8 + red + eps), -1, 1)
                else:
                    data = np.clip((p_b8 - p_b5) / (p_b8 + p_b5 + eps), -1, 1)

                axes[r, 1 + j].imshow(data, cmap='RdYlGn',
                                       vmin=vmins[r], vmax=vmaxs[r])
                if r == 0:
                    axes[r, 1 + j].set_title(name, fontsize=10, fontweight='bold')
                axes[r, 1 + j].axis('off')

        plt.tight_layout()
        plt.savefig(f'complete_values_sample_{i + 1}.png',
                    dpi=150, bbox_inches='tight')
        plt.show()

        # ── Grid 2: Errors ──────────────────────────────────────────────
        fig2, axes2 = plt.subplots(n_rows, n_models,
                                    figsize=(3.0 * n_models, 2.6 * n_rows))
        fig2.suptitle(f'Absolute Error — Sample {i + 1}',
                      fontsize=16, fontweight='bold', y=1.005)

        if n_models == 1:
            axes2 = axes2[:, np.newaxis]

        for r in range(n_rows):
            for j, name in enumerate(model_names):
                pred = predictions_dict[name][i]
                p_b8 = np.clip(pred[:, :, 3], 0, 1)
                p_b5 = np.clip(pred[:, :, 0], 0, 1)

                if r < 4:
                    pred_map = np.clip(pred[:, :, r], 0, 1)
                elif r == 4:
                    pred_map = np.clip((p_b8 - red) / (p_b8 + red + eps), -1, 1)
                else:
                    pred_map = np.clip((p_b8 - p_b5) / (p_b8 + p_b5 + eps), -1, 1)

                error = np.abs(gt_maps[r] - pred_map) * mask
                mae   = np.sum(error) / (np.sum(mask) + 1e-8)

                im2 = axes2[r, j].imshow(error, cmap=ERR_CMAP,
                                          vmin=0, vmax=evmaxs[r])
                if r == 0:
                    axes2[r, j].set_title(name, fontsize=10, fontweight='bold')
                axes2[r, j].text(0.98, 0.02, f'MAE {mae:.4f}',
                                 transform=axes2[r, j].transAxes, fontsize=8,
                                 color='white', ha='right', va='bottom',
                                 bbox=dict(facecolor='black', alpha=0.6, pad=2,
                                           boxstyle='round,pad=0.3'))
                axes2[r, j].axis('off')

            axes2[r, 0].set_ylabel(row_labels[r], fontsize=11, fontweight='bold',
                                    rotation=0, labelpad=60, va='center')
            axes2[r, 0].yaxis.set_visible(True)
            axes2[r, 0].tick_params(left=False, labelleft=False)

        cbar = fig2.colorbar(im2, ax=axes2[:, -1].tolist(),
                              fraction=0.025, pad=0.02)
        cbar.set_label('Absolute Error', fontsize=10)
        plt.tight_layout()
        plt.savefig(f'complete_error_sample_{i + 1}.png',
                    dpi=150, bbox_inches='tight')
        plt.show()
        print(f"Saved sample {i + 1}")

print("Complete comparison function defined")

In [ ]:
# Generate complete comparison figures
print("Generating publication-ready comparison grids...\n")
visualize_complete_comparison(all_predictions, viz_rgb, viz_nir, viz_mask, num_samples=4)